# 04 - Pitcher fit scores and free-agent targets

Each pitcher's mix of archetypes (weighted by balls in play) is combined with each park's archetype effects.

- `park_effect_pts`: expected wOBA points on contact vs. league average at that park (negative helps the pitcher)
- `fit_score`: how much better this park suits the pitcher than the average park does (higher is a better fit); use this to rank targets
- `coverage`: share of the pitcher's mix that had enough data at that park

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.config import PROCESSED, INTERIM
from src.park_effects import contact_frame
from src.fit import pitcher_bip_mix, park_fit, best_pitchers_for_park

df = pd.read_parquet(INTERIM / "pitches_archetyped.parquet")
contact = contact_frame(df)
effects = pd.read_csv(PROCESSED / "park_archetype_effects.csv")

names = contact.drop_duplicates("pitcher").set_index("pitcher")["player_name"]
mix = pitcher_bip_mix(contact, min_bip=150)
fit = park_fit(mix, effects)
fit.head()

## Best-fit pitchers for one park

In [ ]:
best_pitchers_for_park(fit, "Coors Field", names)

## Free-agent class

Fill in `data/processed/free_agents_2026.csv` with the current free-agent pitchers (columns `mlbam_id`, `name`, `notes`). Get MLBAM IDs with `pybaseball.playerid_lookup(last, first)` and use the `key_mlbam` column.

In [ ]:
fa = pd.read_csv(PROCESSED / "free_agents_2026.csv")
if fa.empty:
    print("Add free agents to data/processed/free_agents_2026.csv first.")
else:
    fa_fit = fit.merge(fa, left_on="pitcher", right_on="mlbam_id")
    top = (fa_fit.sort_values("fit_score", ascending=False)
                 .groupby("name").head(3)
                 .sort_values(["name", "fit_score"], ascending=[True, False]))
    top[["name", "park", "fit_score", "coverage"]]

## Backtest plan

The claim to test: knowing a pitcher's archetype mix and the park's archetype effects predicts how his results change when he changes parks better than a plain park factor does.

1. List pitchers who changed home parks (free agency or trade) between consecutive seasons in your window.
2. For each, recompute the park effects **without that pitcher's own pitches** so he cannot predict himself. Use the archetype mix from the seasons before the move.
3. Prediction: his `park_effect_pts` at the new park minus at the old park. Baseline: the same difference using one park-wide number (average gap by park) instead of archetype-level effects.
4. Outcome: the change in his own `woba_gap` on contact from before to after the move.
5. Compare correlation and error for the model vs. the baseline. Report the result honestly, including if the edge is small. Moves are rare, so samples will be small.

In [ ]:
# TODO: backtest (see plan above)

## Commit

```
git add notebooks/04_fa_targets.ipynb data/processed
git commit -m "pitcher fit scores and free-agent targets"
git push
```